# 🦛 Chonkie × 🦙 LlamaIndex
## Document Q&A with Recursive Chunking

_A hands-on tutorial showing how to use Chonkie's `RecursiveChunker` as a native LlamaIndex node parser._

[![GitHub](https://img.shields.io/badge/GitHub-chonkie--inc%2Fchonkie-181717?logo=github)](https://github.com/chonkie-inc/chonkie)
[![Docs](https://img.shields.io/badge/Docs-docs.chonkie.ai-blue)](https://docs.chonkie.ai)
[![Discord](https://img.shields.io/badge/Discord-Join%20Community-5865F2?logo=discord)](https://discord.gg/rYYp6DC4cv)

</div>

---

## What you'll learn

In this tutorial you will:

1. Install and configure the `llama-index-node-parser-chonkie` integration
2. Load a real Wikipedia article as a LlamaIndex `Document`
3. Build a RAG query engine using Chonkie's **recursive chunking** strategy
4. Inspect the produced chunks and run queries against the index

---

### Why recursive chunking?

Recursive chunking is Chonkie's best general-purpose strategy. It tries to split text at natural boundaries — paragraphs first, then sentences, then tokens — so chunks stay coherent without requiring an embedding model at indexing time. It's fast, requires no extra dependencies, and works well across most document types.


## 1. Installation


In [ ]:
# Install all required packages
!pip install -q llama-index-node-parser-chonkie
!pip install -q llama-index-embeddings-huggingface llama-index-llms-openrouter
!pip install -q sentence-transformers wikipedia


## 2. Set API Key

This notebook uses [OpenRouter](https://openrouter.ai) as the LLM backend. Get a free key at https://openrouter.ai/keys.


In [ ]:
import os
from getpass import getpass

# Set your OpenRouter API key
# Get yours at: https://openrouter.ai/keys
os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")


Enter your OpenRouter API key: ··········


## 3. Load a Wikipedia Document

We'll use the Wikipedia article on the **History of Artificial Intelligence** — it's long (~50,000 characters), densely structured, and a great stress-test for any chunker.


In [ ]:
import wikipedia
from llama_index.core import Document

WIKI_PAGE = "History of artificial intelligence"

page = wikipedia.page(WIKI_PAGE, auto_suggest=False)
doc = Document(
    text=page.content,
    metadata={"title": page.title, "url": page.url}
)

print(f"Loaded: '{page.title}'")
print(f"Length: {len(page.content):,} characters")
print(f"\nFirst 500 chars:\n{page.content[:500]}")


Loaded: 'History of artificial intelligence'
Length: 94,729 characters

First 500 chars:
The history of artificial intelligence (AI) began in antiquity, with myths, stories, and rumors of artificial beings endowed with intelligence or consciousness by master craftsmen. The study of logic and formal reasoning from antiquity to the present led directly to the invention of the programmable digital computer in the 1940s, a machine based on abstract mathematical reasoning. This device and the ideas behind it inspired scientists to begin discussing the possibility of building an electroni


## 4. The Chonkie Node Parser

The `Chunker` class from `llama_index.node_parser.chonkie` wraps any Chonkie chunking strategy as a standard LlamaIndex node parser.

You can initialize it in two ways:

**Option A: string alias (recommended for most cases):**
```python
from llama_index.node_parser.chonkie import Chunker
parser = Chunker("recursive", chunk_size=512)
```

**Option B: pass a Chonkie instance directly (for fine-grained config):**
```python
from chonkie import RecursiveChunker
from llama_index.node_parser.chonkie import Chunker

chonkie_chunker = RecursiveChunker(chunk_size=512)
parser = Chunker(chonkie_chunker)
```


### Configure global LlamaIndex settings


In [ ]:
from llama_index.llms.openrouter import OpenRouter
from llama_index.core import Settings

# Using OpenRouter as LLM backend
Settings.llm = OpenRouter(
    model="mistralai/mistral-7b-instruct",
    api_key=os.environ["OPENROUTER_API_KEY"],
)


## 5. Build an Index with Recursive Chunking

We chunk the document using `RecursiveChunker` and embed it locally using `sentence-transformers`.


In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.node_parser.chonkie import Chunker

CHUNK_SIZE = 512

# Use sentence-transformers locally for embeddings
Settings.embed_model = HuggingFaceEmbedding(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

pipeline = IngestionPipeline(
    transformations=[
        Chunker("recursive", chunk_size=CHUNK_SIZE),
    ]
)

nodes = pipeline.run(documents=[doc])
print(f"Produced {len(nodes)} nodes with RecursiveChunker (chunk_size={CHUNK_SIZE})")

index = VectorStoreIndex(nodes)


## 6. Inspect the Chunks

Let's peek at the first few chunks to see what recursive chunking actually produces:


In [ ]:
from IPython.display import display, Markdown

output = "### First 5 chunks (RecursiveChunker)\n\n"
for i, node in enumerate(nodes[:5]):
    text = node.text.replace("\n", " ")
    output += f"**Chunk {i+1}** ({len(node.text)} chars)\n\n{text}\n\n---\n\n"

display(Markdown(output))


### First 5 chunks (RecursiveChunker)

**Chunk 1** (384 chars)

The history of artificial intelligence (AI) began in antiquity, with myths, stories, and rumors of artificial beings endowed with intelligence or consciousness by master craftsmen. The study of logic and formal reasoning from antiquity to the present led directly to the invention of the programmable digital computer in the 1940s, a machine based on abstract mathematical reasoning. 

---

**Chunk 2** (125 chars)

This device and the ideas behind it inspired scientists to begin discussing the possibility of building an electronic brain. 

---

**Chunk 3** (364 chars)

The field of AI research was founded at a workshop held on the campus of Dartmouth College in 1956. Attendees of the workshop became the leaders of AI research for decades. Many of them predicted that machines as intelligent as humans would exist within a generation. The U.S. government provided millions of dollars with the hope of making this vision come true. 

---

**Chunk 4** (282 chars)

Eventually, it became obvious that researchers had grossly underestimated the difficulty of this feat. In 1974, criticism from James Lighthill and pressure from the U.S. Congress led the U.S. and British Governments to stop funding undirected research into artificial intelligence. 

---

**Chunk 5** (369 chars)

Seven years later, a visionary initiative by the Japanese Government and the success of expert systems  reinvigorated investment in AI, and by the late 1980s, the industry had grown into a billion-dollar enterprise. However, investors' enthusiasm waned in the 1990s, and the field was criticized in the press and avoided by industry (a period known as an "AI winter"). 

---



## 7. Query the Index

Now let's run some questions against the index:


In [ ]:
import textwrap

engine = index.as_query_engine(similarity_top_k=3)

questions = [
    "How did the AI winters affect research funding and progress?",
    "What role did deep learning play in the modern AI renaissance?",
    "Summarize the key turning points in AI history.",
]

for question in questions:
    print("=" * 65)
    print(f"Q: {question}")
    print("=" * 65)
    response = engine.query(question)
    wrapped = textwrap.fill(str(response).strip(), width=150)
    print(wrapped)
    print()


Q: How did the AI winters affect research funding and progress?
The AI winters were marked by significant declines in research funding and slowed progress due to widespread disappointment in the field. After
periods of high expectations and substantial investments—particularly following the hype around expert systems in the 1970s and the rapid
commercialization of AI in the late 1980s—many projects failed to deliver on their promises, leading to financial setbacks and reduced enthusiasm.
Companies and governments scaled back support, as the lack of clear, sustainable profitability models and the gap between early claims and actual
results made it difficult to justify continued funding. This shift created a downturn in both academic and industry interest, with fewer resources
allocated to AI research and development, ultimately stalling advancements for several years. However, perspectives on this period vary, with some
experts later arguing that the challenges were not as severe as the

## 8. Passing a Chonkie Instance Directly

For full parameter control, configure a `RecursiveChunker` instance and pass it straight to the `Chunker` wrapper:


In [ ]:
from chonkie import RecursiveChunker
from llama_index.node_parser.chonkie import Chunker

# Full control over chunker parameters
chonkie_chunker = RecursiveChunker(
    chunk_size=512,
    # chunk_overlap=50,  # uncomment to add overlap
)

parser = Chunker(chonkie_chunker)
nodes = parser.get_nodes_from_documents([doc])
print(f"Produced {len(nodes)} nodes with configured RecursiveChunker")


Produced 356 nodes with configured RecursiveChunker


## Summary

Here's what we covered:

- **One import** drops Chonkie's recursive chunker into any LlamaIndex pipeline
- **String alias** (`"recursive"`) makes initialization simple
- **Direct instance passing** gives full configuration control when needed
- **IngestionPipeline-native** — Chonkie fits seamlessly into production workflows

---

### What's next?

- 📦 **Install**: `pip install llama-index-node-parser-chonkie`
- 📚 **Chonkie docs**: [docs.chonkie.ai](https://docs.chonkie.ai)
- 🌟 **Star Chonkie**: [github.com/chonkie-inc/chonkie](https://github.com/chonkie-inc/chonkie)
- 💬 **Join Discord**: [discord.gg/rYYp6DC4cv](https://discord.gg/rYYp6DC4cv)
